In [ ]:
import spikeinterface.full as si
import spikeinterface.extractors as se

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import os

import warnings
warnings.simplefilter("ignore")

# %matplotlib widget
%matplotlib inline

# Preprocessing via SpikeInterface

## Load raw data from OpenEphys

If not all channels but only specific channels are selected during recording,
you can change the name of the setting.txt to show the metadata

In [3]:
ap_path = "/home/user/DATA/Sally/Npx_WT_prelim_2026/2026_02_13/2026-02-13_14-07-50/Record Node 101/experiment1/recording1"
full_raw_rec = si.read_openephys(ap_path, stream_name="Record Node 101#OneBox-100.ProbeA")

print(full_raw_rec)

OpenEphysBinaryRecordingExtractor: 38 channels - 30.0kHz - 1 segments - 167,423,504 samples 
                                   5,580.78s (1.55 hours) - int16 dtype - 11.85 GiB


## Property

`Recording` objects can have *properties*. A property is a piece of information attached to a channel, e.g. group or location.

We can check which properties are in the extractor as follows:

In [8]:
print("Properties:\n", list(full_raw_rec.get_property_keys()))

Properties:
 ['gain_to_uV', 'offset_to_uV', 'physical_unit', 'gain_to_physical_unit', 'offset_to_physical_unit', 'channel_name']


In [4]:
full_raw_rec.channel_ids

array(['CH34', 'CH35', 'CH36', 'CH37', 'CH50', 'CH51', 'CH52', 'CH53',
       'CH66', 'CH67', 'CH68', 'CH69', 'CH82', 'CH83', 'CH84', 'CH85',
       'CH98', 'CH99', 'CH100', 'CH101', 'CH114', 'CH115', 'CH116',
       'CH117', 'CH130', 'CH131', 'CH132', 'CH133', 'CH146', 'CH147',
       'CH148', 'CH149', 'CH162', 'CH163', 'CH164', 'CH178', 'CH179',
       'CH180'], dtype='<U64')

In [9]:
print(full_raw_rec.get_annotation_keys())

['is_filtered', 'experiment_name']


In [5]:
si.recording_extractor_full_dict

{'binaryfolder': spikeinterface.core.binaryfolder.BinaryFolderRecording,
 'binary': spikeinterface.core.binaryrecordingextractor.BinaryRecordingExtractor,
 'zarr': spikeinterface.core.zarrextractors.ZarrRecordingExtractor,
 'numpy': spikeinterface.core.numpyextractors.NumpyRecording,
 'shybrid': spikeinterface.extractors.shybridextractors.SHYBRIDRecordingExtractor,
 'mda': spikeinterface.extractors.mdaextractors.MdaRecordingExtractor,
 'nwb': spikeinterface.extractors.nwbextractors.NwbRecordingExtractor,
 'nwbtimeseries': spikeinterface.extractors.nwbextractors.NwbTimeSeriesExtractor,
 'compressedbinaryibl': spikeinterface.extractors.cbin_ibl.CompressedBinaryIblExtractor,
 'ibl': spikeinterface.extractors.iblextractors.IblRecordingExtractor,
 'mcsh5': spikeinterface.extractors.mcsh5extractors.MCSH5RecordingExtractor,
 'sinapsresearchplatform': spikeinterface.extractors.sinapsrecordingextractors.SinapsResearchPlatformRecordingExtractor,
 'sinapsresearchplatformh5': spikeinterface.extrac

## Bandpass filter and CMR

Below, we bandpass filter the recording and apply common median reference to the original recording:

In [10]:
recording_f = si.bandpass_filter(full_raw_rec, freq_min=300, freq_max=9000)

In [11]:
print(recording_f)

BandpassFilterRecording: 38 channels - 30.0kHz - 1 segments - 167,423,504 samples 
                         5,580.78s (1.55 hours) - int16 dtype - 11.85 GiB


Let's now apply Common Median Reference (CMR):

In [12]:
recording_cmr = si.common_reference(recording_f, reference='global', operator='median')

In [13]:
print(recording_cmr)
recording_cmr.get_total_duration()


CommonReferenceRecording: 38 channels - 30.0kHz - 1 segments - 167,423,504 samples 
                          5,580.78s (1.55 hours) - int16 dtype - 11.85 GiB


np.float64(5580.783466666667)

In [14]:
import numpy as np
import matplotlib.pyplot as plt

rec = recording_cmr  # or recording_f

fs = rec.get_sampling_frequency()
nch = rec.get_num_channels()
dur = rec.get_total_duration()

print("fs:", fs)
print("n_channels:", nch)
print("duration_s:", dur)

# Some extractors/wrappers have non-zero time offsets
try:
    print("start_time:", rec.get_start_time())
except Exception as e:
    print("get_start_time() not available:", repr(e))

# Check if raw frame fetch works at all (frame-based, bypasses widget time conversion)
chunk0 = rec.get_traces(start_frame=0, end_frame=min(int(fs * 0.1), rec.get_num_samples()))
print("chunk0 shape:", chunk0.shape)

# Check your requested 10.0-10.1 s window in frame coordinates
sf = int(10.0 * fs)
ef = int(10.1 * fs)
chunk10 = rec.get_traces(start_frame=sf, end_frame=min(ef, rec.get_num_samples()))
print("chunk10 shape:", chunk10.shape)

fs: 30000.0
n_channels: 38
duration_s: 5580.783466666667
start_time: 31.483133333333335
chunk0 shape: (3000, 38)
chunk10 shape: (3000, 38)


We can plot the traces after applying CMR:

In [15]:
%matplotlib widget
w = si.plot_traces({"filtered": recording_f, "common": recording_cmr}, mode='line',
                   time_range=[50, 50.1], backend="ipywidgets")

AppLayout(children=(TimeSlider(children=(Dropdown(description='segment', options=(0,), value=0), Button(icon='…

## Remove bad channels

In [16]:
bad_channel_ids, channel_labels = si.detect_bad_channels(recording_f, method='std')
print('bad_channel_ids', bad_channel_ids)
print('channel_labels', channel_labels)

bad_channel_ids []
channel_labels ['good' 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good'
 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good'
 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good'
 'good' 'good' 'good' 'good' 'good' 'good' 'good' 'good']


In [17]:
recording_good_channels_f = recording_f.remove_channels(bad_channel_ids)
recording_good_channels = si.common_reference(recording_good_channels_f, reference='global', operator='median')

print(recording_good_channels)
print(recording_good_channels.channel_ids)

CommonReferenceRecording: 38 channels - 30.0kHz - 1 segments - 167,423,504 samples 
                          5,580.78s (1.55 hours) - int16 dtype - 11.85 GiB
['CH34' 'CH35' 'CH36' 'CH37' 'CH50' 'CH51' 'CH52' 'CH53' 'CH66' 'CH67'
 'CH68' 'CH69' 'CH82' 'CH83' 'CH84' 'CH85' 'CH98' 'CH99' 'CH100' 'CH101'
 'CH114' 'CH115' 'CH116' 'CH117' 'CH130' 'CH131' 'CH132' 'CH133' 'CH146'
 'CH147' 'CH148' 'CH149' 'CH162' 'CH163' 'CH164' 'CH178' 'CH179' 'CH180']


In [18]:
w = si.plot_traces(recording_good_channels, mode='line',
                   time_range=[50, 50.1],
                   backend="ipywidgets")

AppLayout(children=(TimeSlider(children=(Dropdown(description='segment', options=(0,), value=0), Button(icon='…

## Saving and Reload

`job_kwargs` controls how SI splits data into chunks and runs work in parallel.

In [19]:
job_kwargs = dict(n_jobs=4, chunk_duration="1s", progress_bar=True)

Save the file in SpikeInterface format

In [22]:
recording_saved = recording_cmr.save(
    folder="/home/user/DATA/Sally/Npx_WT_prelim_2026/2026_02_13/preprocessed",
    format="binary", # SI BinaryFolderRecording
    **job_kwargs
)


write_binary_recording 
engine=process - n_jobs=4 - samples_per_chunk=30,000 - chunk_memory=2.17 MiB - total_memory=8.70 MiB - chunk_duration=1.00s


write_binary_recording (workers: 4 processes):   0%|          | 0/5581 [00:00<?, ?it/s]

In [24]:
recording_saved

BinaryFolderRecording: 38 channels - 30.0kHz - 1 segments - 167,423,504 samples 
                       5,580.78s (1.55 hours) - int16 dtype - 11.85 GiB

After saving the SI object, we can easily load it back in a new session:

In [ ]:
#recording_loaded = si.load_extractor("/home/user/DATA/Sally/Npx_WT_prelim_2026/2026_02_13/preprocessed")

Export bin file so that kilosort can use

In [23]:
si.write_binary_recording(recording_cmr,
                          file_paths="/home/user/DATA/Sally/Npx_WT_prelim_2026/2026_02_13/preprocessed_ks.bin",
                          dtype="int16",
                          **job_kwargs)

write_binary_recording (workers: 4 processes):   0%|          | 0/5581 [00:00<?, ?it/s]

# Kilosort and Manual Curation (in Phy)

→ Build `.bin` 

→ run `Kilosort` 

→ curate in `Phy` 

→ re-import into `SpikeInterface` for saving the good units.

when specific channels are recorded, the default setting of NP1 or NP2 probe configuration in kilosort cannot be used.

Here is a way to build your own probe setting for kilosort

In [30]:
import re
import numpy as np

# Put your 38 channel IDs here IN THE SAME ORDER AS THE BIN ROWS
ch_names = ['CH34', 'CH35', 'CH36', 'CH37', 'CH50', 'CH51', 'CH52', 'CH53', 'CH66', 'CH67',
 'CH68', 'CH69', 'CH82', 'CH83', 'CH84', 'CH85', 'CH98', 'CH99', 'CH100', 'CH101',
 'CH114', 'CH115', 'CH116', 'CH117', 'CH130', 'CH131', 'CH132', 'CH133', 'CH146',
 'CH147', 'CH148', 'CH149', 'CH162', 'CH163', 'CH164', 'CH178' , 	' CH179' , 	' CH180']

idx = np.array([int(re.findall(r"\d+", s)[0]) for s in ch_names], dtype=int)

# NP2-like approx (2 columns, 32um apart, 15um vertical pitch)
xc = (idx % 2) * 32
yc = (idx // 2) * 15

chanMap = list(range(38))        # ALWAYS 0..37 for a 38-ch bin
kcoords = [0] * 38               # single shank/group

print('xc =', xc.tolist())
print('yc =', yc.tolist())
print('chanMap =', chanMap)
print('kcoords =', kcoords)

xc = [0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 32, 0, 0, 32, 0]
yc = [255, 255, 270, 270, 375, 375, 390, 390, 495, 495, 510, 510, 615, 615, 630, 630, 735, 735, 750, 750, 855, 855, 870, 870, 975, 975, 990, 990, 1095, 1095, 1110, 1110, 1215, 1215, 1230, 1335, 1335, 1350]
chanMap = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
kcoords = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Then in the GUI probe dialog:
- Name: test
- X-coordinates: paste the xc = [...] list
- Y-coordinates: paste the yc = [...] list (this is what encodes gaps)
- kcoords: leave blank for single channel
- chanMap: `list(range(38))`

This aligns with Kilosort’s required probe dictionary fields (chanMap, xc, yc, kcoords, n_chan).

# Postprocessing and Export Data

After curating in Phy, the “ground truth” for spike trains is in three files inside the Phy/Kilosort output folder:

- `spike_times.npy` (spike sample indices)
- `spike_clusters.npy` (cluster/unit id per spike)
- `cluster_group.tsv` (your labels: good / mua / noise) 

In [ ]:
phy_folder = Path('/home/user/DATA/Sally/Npx_WT_prelim_2026/2026_02_13/kilosort4')

sorting = se.read_phy(phy_folder)

cg = pd.read_csv(phy_folder / 'cluster_group.tsv', sep='\t')
good_ids = cg.loc[cg['group'] == 'good', 'cluster_id'].to_numpy(dtype=int)

print('number of good units:', len(good_ids))
print('good units:', good_ids)

number of good units: 14
good units: [ 4  7 17 18 22 33 53 61 62 71 86 94 95 97]


In [39]:
rows = []
for uid in good_ids:
    st = sorting.get_unit_spike_train(unit_id=uid, segment_index=0)  # continuous recordings have only one segment
    rows.append(np.column_stack([st.astype(np.int64),
                                 np.full(st.size, uid, dtype=np.int32)]))

good_spikes = np.concatenate(rows, axis=0) if rows else np.zeros((0, 2), dtype=np.int64)
np.save(phy_folder / "good_spikes_sample_unit.npy", good_spikes)